# 01C — Second-pass family rebuild using the fresh 01B findings

**NO MODEL TRAINING.**

Your first clean split passed all previously known relationships, but the fresh 01B audit discovered additional
high-confidence near-duplicate pairs. This notebook:

1. Reads the first clean dataset.
2. Preserves **all family links from the first cleaning pass**.
3. Adds **all newly confirmed 01B near-duplicate links**.
4. Quarantines any newly formed family containing multiple class labels.
5. Rebuilds Train / Validation / Test at the expanded family level.
6. Writes a completely new folder, leaving both the original dataset and first clean version untouched.

Output:
`Cataract/Data_Clean_LeakageControlled_v2`

In [1]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, shutil, random, json, hashlib
import numpy as np
import pandas as pd

SEED=42
random.seed(SEED)
np.random.seed(SEED)

PROJECT=Path('/content/drive/MyDrive/Cataract')

SOURCE=PROJECT/'Data_Clean_LeakageControlled'
OLD_MANIFEST=PROJECT/'FINAL_REVISION_2026_08'/'clean_split_audit'/'split_manifest.csv'
NEW_PAIRS=PROJECT/'FINAL_REVISION_2026_08'/'clean_split_audit'/'fresh_reaudit'/'CONFIRMED_CROSS_PARTITION_NEAR_DUPLICATES.csv'

OUTROOT=PROJECT/'Data_Clean_LeakageControlled_v2'
REPORT=PROJECT/'FINAL_REVISION_2026_08'/'clean_split_audit_v2'
QUARANTINE=PROJECT/'FINAL_REVISION_2026_08'/'clean_split_quarantine_v2'

REPORT.mkdir(parents=True,exist_ok=True)
QUARANTINE.mkdir(parents=True,exist_ok=True)

assert SOURCE.exists(), SOURCE
assert OLD_MANIFEST.exists(), OLD_MANIFEST
assert NEW_PAIRS.exists(), NEW_PAIRS

pairs=pd.read_csv(NEW_PAIRS)
old_manifest=pd.read_csv(OLD_MANIFEST)

print('Fresh confirmed 01B pairs:',len(pairs))
print('Old manifest rows:',len(old_manifest))
display(pairs.head())

Mounted at /content/drive
Fresh confirmed 01B pairs: 514
Old manifest rows: 13669


,query_path,query_split,query_class,ref_path,ref_split,ref_class,phash_distance,candidate_type,sift_good,sift_inliers,sift_inlier_ratio,confirmed_strict
0,Validation/Cataract/cat_0_1012.jpg,Validation,Cataract,Train/Cataract/cat_0_1205.jpg,Train,Cataract,12,same_class,82,74,0.902439,True
1,Validation/Cataract/cat_0_1280.jpg,Validation,Cataract,Train/Cataract/cat_0_8288.jpg,Train,Cataract,6,same_class,58,52,0.896552,True
2,Validation/Cataract/cat_0_1504.jpg,Validation,Cataract,Train/Cataract/cat_0_146.jpg,Train,Cataract,10,same_class,49,43,0.877551,True
3,Validation/Cataract/cat_0_1553.jpg,Validation,Cataract,Train/Cataract/cat_0_7013.jpg,Train,Cataract,12,same_class,142,129,0.908451,True
4,Validation/Cataract/cat_0_1589.jpg,Validation,Cataract,Train/Cataract/cat_0_621.jpg,Train,Cataract,8,same_class,605,598,0.988430,True


In [2]:
CLASS_ORDER=['Cataract','Normal','Not Eye']
SPLITS=['Train','Validation','Test']
valid_ext={'.jpg','.jpeg','.png','.bmp','.webp'}

# Current-v1 inventory
rows=[]
for split in SPLITS:
    for cls in CLASS_ORDER:
        d=SOURCE/split/cls
        for p in sorted(d.glob('*')):
            if p.is_file() and p.suffix.lower() in valid_ext:
                rows.append({
                    'v1_path':p.relative_to(SOURCE).as_posix(),
                    'v1_split':split,
                    'class':cls,
                    'filename':p.name,
                    'source_abs':str(p)
                })
files=pd.DataFrame(rows)
print(files.groupby(['v1_split','class']).size())
print('TOTAL v1:',len(files))

v1_split    class   
Test        Cataract     857
            Normal       979
            Not Eye      756
Train       Cataract    2932
            Normal      3348
            Not Eye     2584
Validation  Cataract     722
            Normal       825
            Not Eye      636
dtype: int64
TOTAL v1: 13639


In [3]:
class DSU:
    def __init__(self,items):
        self.p={x:x for x in items}
        self.r={x:0 for x in items}
    def find(self,x):
        while self.p[x]!=x:
            self.p[x]=self.p[self.p[x]]
            x=self.p[x]
        return x
    def union(self,a,b):
        if a not in self.p or b not in self.p:
            return
        a,b=self.find(a),self.find(b)
        if a==b:return
        if self.r[a]<self.r[b]:a,b=b,a
        self.p[b]=a
        if self.r[a]==self.r[b]:self.r[a]+=1

all_paths=set(files.v1_path)
dsu=DSU(all_paths)

# ---------------------------------------------------------
# A. Preserve ALL family relationships from the first pass.
# ---------------------------------------------------------
inc=old_manifest[old_manifest.status=='INCLUDED'].copy()

def path_after_clean_root(x):
    x=str(x).replace('\\','/')
    marker='Data_Clean_LeakageControlled/'
    if marker in x:
        return x.split(marker,1)[1]
    return None

inc['v1_path']=inc['new_path'].map(path_after_clean_root)
inc=inc[inc.v1_path.isin(all_paths)]

for fid,g in inc.groupby('family_id'):
    paths=g.v1_path.tolist()
    for x in paths[1:]:
        dsu.union(paths[0],x)

print('Preserved original family groups:',inc.family_id.nunique())

# ---------------------------------------------------------
# B. Add ALL newly confirmed 01B relations.
# ---------------------------------------------------------
for _,r in pairs.iterrows():
    dsu.union(r['query_path'],r['ref_path'])

files['family_root']=files.v1_path.map(dsu.find)
roots={r:i for i,r in enumerate(sorted(files.family_root.unique()),1)}
files['family_id_v2']=files.family_root.map(lambda x:f'V2F{roots[x]:05d}')

family_classes=files.groupby('family_id_v2')['class'].agg(lambda x:sorted(set(x))).to_dict()
conflicted={fid for fid,cs in family_classes.items() if len(cs)>1}
files['label_conflict_v2']=files.family_id_v2.isin(conflicted)

print('V2 total families:',files.family_id_v2.nunique())
print('V2 mixed-label families:',len(conflicted))
if conflicted:
    display(files[files.label_conflict_v2].sort_values(['family_id_v2','class','v1_path']).head(100))

Preserved original family groups: 13259
V2 total families: 12759
V2 mixed-label families: 6


,v1_path,v1_split,class,filename,source_abs,family_root,family_id_v2,label_conflict_v2
11130,Test/Cataract/cat_0_1851.jpg,Test,Cataract,cat_0_1851.jpg,/content/drive/MyDrive/Cataract/Data_Clean_Lea...,Test/Cataract/cat_0_1851.jpg,V2F00079,True
1934,Train/Cataract/cat_0_6863.jpg,Train,Cataract,cat_0_6863.jpg,/content/drive/MyDrive/Cataract/Data_Clean_Lea...,Test/Cataract/cat_0_1851.jpg,V2F00079,True
5022,Train/Normal/cat_0_6538.jpg,Train,Normal,cat_0_6538.jpg,/content/drive/MyDrive/Cataract/Data_Clean_Lea...,Test/Cataract/cat_0_1851.jpg,V2F00079,True
11207,Test/Cataract/cat_0_274.jpg,Test,Cataract,cat_0_274.jpg,/content/drive/MyDrive/Cataract/Data_Clean_Lea...,Test/Cataract/cat_0_7451.jpg,V2F00591,True
11661,Test/Cataract/cat_0_7451.jpg,Test,Cataract,cat_0_7451.jpg,/content/drive/MyDrive/Cataract/Data_Clean_Lea...,Test/Cataract/cat_0_7451.jpg,V2F00591,True
9904,Validation/Normal/cat_0_4560.jpg,Validation,Normal,cat_0_4560.jpg,/content/drive/MyDrive/Cataract/Data_Clean_Lea...,Test/Cataract/cat_0_7451.jpg,V2F00591,True
2213,Train/Cataract/cat_0_7813.jpg,Train,Cataract,cat_0_7813.jpg,/content/drive/MyDrive/Cataract/Data_Clean_Lea...,Test/Normal/cat_0_7497.jpg,V2F01521,True
12629,Test/Normal/cat_0_7497.jpg,Test,Normal,cat_0_7497.jpg,/content/drive/MyDrive/Cataract/Data_Clean_Lea...,Test/Normal/cat_0_7497.jpg,V2F01521,True
8935,Validation/Cataract/cat_0_1822.jpg,Validation,Cataract,cat_0_1822.jpg,/content/drive/MyDrive/Cataract/Data_Clean_Lea...,Validation/Cataract/cat_0_1822.jpg,V2F10758,True
3250,Train/Normal/cat_0_1837.jpg,Train,Normal,cat_0_1837.jpg,/content/drive/MyDrive/Cataract/Data_Clean_Lea...,Validation/Cataract/cat_0_1822.jpg,V2F10758,True


## Split expanded families

All members of an expanded family are assigned together.

Target proportions remain approximately:
- Train 65%
- Validation 16%
- Test 19%

In [4]:
eligible=files[~files.label_conflict_v2].copy()

fam=eligible.groupby('family_id_v2').agg(
    class_name=('class','first'),
    n_files=('v1_path','size')
).reset_index()

TARGET={'Train':0.65,'Validation':0.16,'Test':0.19}
rng=np.random.default_rng(SEED)
assignment={}

for cls in CLASS_ORDER:
    fc=fam[fam.class_name==cls].copy()
    fc['rand']=rng.random(len(fc))
    fc=fc.sort_values(['n_files','rand'],ascending=[False,True])

    total=int(fc.n_files.sum())
    targets={k:TARGET[k]*total for k in TARGET}
    counts={k:0 for k in TARGET}

    for _,r in fc.iterrows():
        deficits={k:(targets[k]-counts[k])/max(targets[k],1) for k in TARGET}
        dest=max(deficits,key=deficits.get)
        assignment[r.family_id_v2]=dest
        counts[dest]+=int(r.n_files)

    print(cls,'total=',total,'actual=',counts)

eligible['new_split_v2']=eligible.family_id_v2.map(assignment)
display(eligible.groupby(['new_split_v2','class']).size().unstack(fill_value=0))

Cataract total= 4502 actual= {'Train': 2926, 'Validation': 721, 'Test': 855}
Normal total= 5146 actual= {'Train': 3344, 'Validation': 824, 'Test': 978}
Not Eye total= 3976 actual= {'Train': 2584, 'Validation': 636, 'Test': 756}


class,Cataract,Normal,Not Eye
new_split_v2,,,
Test,855,978,756
Train,2926,3344,2584
Validation,721,824,636


In [5]:
if OUTROOT.exists():
    raise RuntimeError(
        f'STOP: {OUTROOT} already exists. Rename/delete it intentionally before rebuilding.'
    )

for split in SPLITS:
    for cls in CLASS_ORDER:
        (OUTROOT/split/cls).mkdir(parents=True,exist_ok=True)

manifest=[]

for _,r in files.iterrows():
    src=Path(r.source_abs)

    if r.label_conflict_v2:
        dest=QUARANTINE/r.family_id_v2/r['class']/r.filename
        dest.parent.mkdir(parents=True,exist_ok=True)
        shutil.copy2(src,dest)
        status='QUARANTINED_LABEL_CONFLICT_V2'
        newsp='Quarantine'
        newpath=str(dest)
    else:
        newsp=assignment[r.family_id_v2]
        dest=OUTROOT/newsp/r['class']/r.filename
        if dest.exists():
            dest=dest.with_name(f'{dest.stem}__{r.family_id_v2}{dest.suffix}')
        shutil.copy2(src,dest)
        status='INCLUDED'
        newpath=str(dest)

    manifest.append({
        'v1_path':r.v1_path,
        'v1_split':r.v1_split,
        'class':r['class'],
        'family_id_v2':r.family_id_v2,
        'new_split_v2':newsp,
        'status':status,
        'new_path_v2':newpath
    })

manifest=pd.DataFrame(manifest)
manifest.to_csv(REPORT/'split_manifest_v2.csv',index=False)

print('\nV2 status:')
display(manifest.groupby(['status','class']).size().unstack(fill_value=0))
print('\nV2 split:')
display(manifest[manifest.status=='INCLUDED'].groupby(['new_split_v2','class']).size().unstack(fill_value=0))


V2 status:


class,Cataract,Normal,Not Eye
status,,,
INCLUDED,4502,5146,3976
QUARANTINED_LABEL_CONFLICT_V2,9,6,0



V2 split:


class,Cataract,Normal,Not Eye
new_split_v2,,,
Test,855,978,756
Train,2926,3344,2584
Validation,721,824,636


In [6]:
# Verify every newly confirmed 01B pair is now either together or quarantined.
m=manifest.set_index('v1_path')
checks=[]
for _,r in pairs.iterrows():
    a,b=r.query_path,r.ref_path
    if a not in m.index or b not in m.index:
        continue
    sa,sb=m.loc[a,'new_split_v2'],m.loc[b,'new_split_v2']
    bad=sa in SPLITS and sb in SPLITS and sa!=sb
    checks.append({'a':a,'b':b,'split_a':sa,'split_b':sb,'bad':bad})

chk=pd.DataFrame(checks)
chk.to_csv(REPORT/'fresh_01B_pair_verification_after_v2.csv',index=False)

bad=int(chk.bad.sum()) if len(chk) else 0
print('Known 01B confirmed pairs crossing V2 split:',bad)
assert bad==0,'STOP: a known confirmed pair still crosses the V2 split.'

summary={
    'v1_input_files':int(len(files)),
    'v2_included_files':int((manifest.status=='INCLUDED').sum()),
    'v2_quarantined_files':int((manifest.status!='INCLUDED').sum()),
    'v2_families':int(files.family_id_v2.nunique()),
    'v2_conflicted_families':int(len(conflicted)),
    'known_01B_pairs_crossing_v2':bad
}
with open(REPORT/'v2_build_summary.json','w') as f:
    json.dump(summary,f,indent=2)

print(json.dumps(summary,indent=2))
print('\nNEXT: run 01D_Reaudit_Clean_Split_v2.ipynb')

Known 01B confirmed pairs crossing V2 split: 0
{
  "v1_input_files": 13639,
  "v2_included_files": 13624,
  "v2_quarantined_files": 15,
  "v2_families": 12759,
  "v2_conflicted_families": 6,
  "known_01B_pairs_crossing_v2": 0
}

NEXT: run 01D_Reaudit_Clean_Split_v2.ipynb
